# Book Translation Pipeline: English to Hinglish

Translates a long English text (a book or a story) into Hinglish using **Gemma 3 27B**, served locally by **Ollama**. Written for a Colab T4 GPU, but it runs anywhere Ollama does.

The model can only handle a few hundred words well at a time, so the book is cut into chunks. Each chunk is translated in two passes, the output is checked and cleaned, and every finished chunk is written to disk straight away. A run can therefore be stopped at any point and resumed later.

```
book.txt -> clean -> chunk -> for each chunk:
                                 Pass 1: faithful draft
                                 Pass 2: Hinglish rewrite
                                 validate -> retry if the output is bad
                                 post-process -> save to checkpoint
                           -> stitch chunks -> book-level cleanup -> output.txt
```

| # | Section | What it does |
|---|---------|--------------|
| 1 | Configuration and logging | All settings in one cell, plus the logger |
| 2 | Environment | Install Ollama, start the server, pull the model |
| 3 | Book profile | Characters, glossary and style notes for the book being translated |
| 4 | Source text | Load and clean the input file |
| 5 | Chunking | Sentence-aware chunks with a little context from the previous chunk |
| 6 | Prompts | System and user prompts for both passes |
| 7 | Model calls and validation | Streaming generation, loop detection, output checks |
| 8 | Post-processing | Cleanup rules and the repair step |
| 9 | Translating one chunk | Pass 1, Pass 2, retries, fallbacks |
| 10 | Checkpoint and resume | Append-only JSONL file, one line per chunk |
| 11 | Run | Main loop |
| 12 | Output | Stitch the book and write the QA report |

## 1. Configuration and logging

Everything you might want to change is in the next cell. Logging goes to the cell output as the run progresses and is also appended to `run.log` in the output folder.

In [ ]:
import json
import hashlib
import logging
import os
import re
import subprocess
import sys
import time
import urllib.request
from collections import Counter, deque
from difflib import SequenceMatcher
from pathlib import Path

# ---- Model and server
MODEL = "gemma3:27b"
NUM_CTX = 8192                      # context window handed to Ollama
OLLAMA_HOST = "127.0.0.1:11434"

# ---- Input and output
SOURCE_FILE = "book.txt"            # English .txt; on Colab you are asked to upload it if missing
OUTPUT_DIR = Path("translation_output")
# To keep progress across Colab sessions, save to Google Drive instead:
#   from google.colab import drive; drive.mount("/content/drive")
#   OUTPUT_DIR = Path("/content/drive/MyDrive/translation_output")

# ---- Chunking
CHUNK_WORDS = 350                   # target words per chunk
OVERLAP_SENTENCES = 2               # sentences of the previous chunk shown to Pass 1 as context

# ---- Generation
NUM_PREDICT = CHUNK_WORDS * 4       # token cap per call (1400 at 350 words)
MAX_RETRIES = 2                     # extra attempts per pass when validation fails
STORY_NOTES = 5                     # how many previous plot notes Pass 2 sees

PASS1_OPTIONS = dict(temperature=0.15, top_k=20, top_p=0.85, repeat_penalty=1.10)   # faithful
PASS2_OPTIONS = dict(temperature=0.60, top_k=40, top_p=0.90, repeat_penalty=1.15)   # stylistic
REPAIR_OPTIONS = dict(temperature=0.20, top_k=15, top_p=0.80, repeat_penalty=1.10)  # light fixes
# Sampling changes for retries, indexed by attempt number (0 = first try)
RETRY_TWEAKS = [{}, dict(repeat_penalty=1.25), dict(repeat_penalty=1.35, temperature=0.50, top_p=0.80)]

# ---- Validation thresholds
LENGTH_RATIO_RANGE = (0.4, 2.5)     # output words / source words
MAX_ENGLISH_RATIO = 0.10            # English function words per word, whole chunk
REPAIR_ENGLISH_RATIO = 0.12         # same measure, per paragraph: above this the paragraph is repaired

# ---- Run control
TIME_LIMIT_MIN = 270                # stop cleanly after this many minutes (None = no limit)
RESTART = False                     # True = ignore any existing checkpoint and start over
SHOW_PREVIEW = True                 # log the first 120 characters of every finished chunk

# ---- Logging: cell output (live) and run.log
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
log = logging.getLogger("translator")
log.setLevel(logging.INFO)
log.handlers.clear()                # lets this cell be re-run without duplicate lines
log.propagate = False
_fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%H:%M:%S")
for _handler in (logging.StreamHandler(sys.stdout),
                 logging.FileHandler(OUTPUT_DIR / "run.log", encoding="utf-8")):
    _handler.setFormatter(_fmt)
    log.addHandler(_handler)

log.info("config: model=%s chunk=%d words overlap=%d sentences retries=%d",
         MODEL, CHUNK_WORDS, OVERLAP_SENTENCES, MAX_RETRIES)

## 2. Environment

Install the Python client and Ollama, start the server, then make sure the model is downloaded. The model is about 17 GB, so the first download takes a while. A T4 has 15 GB of VRAM, so Ollama offloads a few layers to the CPU; expect roughly 1 to 2 minutes per chunk for both passes.

In [ ]:
!pip install -q ollama
!nvidia-smi -L

In [ ]:
!apt-get install -y -qq zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
os.environ["OLLAMA_HOST"] = OLLAMA_HOST
os.environ["OLLAMA_GPU_OVERHEAD"] = "512000000"     # keep ~0.5 GB of VRAM free on the T4


def ollama_is_up():
    try:
        urllib.request.urlopen(f"http://{OLLAMA_HOST}", timeout=2)
        return True
    except OSError:
        return False


if not ollama_is_up():
    server_log = open(OUTPUT_DIR / "ollama_server.log", "a")
    subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=server_log)
    for _ in range(30):
        if ollama_is_up():
            break
        time.sleep(1)

if ollama_is_up():
    log.info("ollama server is up at %s", OLLAMA_HOST)
else:
    log.error("ollama server did not start - see %s", OUTPUT_DIR / "ollama_server.log")

In [ ]:
import ollama

client = ollama.Client(host=f"http://{OLLAMA_HOST}", timeout=600)


def ensure_model(name):
    """Pull the model unless it is already available locally."""
    try:
        client.show(name)
        log.info("model %s is already available", name)
        return
    except ollama.ResponseError:
        pass
    log.info("pulling %s (about 17 GB)...", name)
    last_logged = -10
    for event in client.pull(name, stream=True):
        total, done = event.get("total"), event.get("completed")
        if total and done is not None:
            pct = int(done / total * 100)
            if pct >= last_logged + 10:
                log.info("  download %d%%", pct)
                last_logged = pct
    log.info("model %s is ready", name)


ensure_model(MODEL)

## 3. Book profile

The only cell that changes from one book to the next. It holds everything the model cannot guess from the text:

- **narrator**: first-person narrators need consistent verb gender (aaya or aayi)
- **characters**: role, how they speak, and who says aap or tum to them
- **keep**: terms that must stay in English exactly as written
- **glossary**: fixed Hinglish renderings for hard or period-specific terms
- **style_notes** and **avoid**: book-level register

Only the glossary terms and characters that actually appear in a chunk are put into that chunk's prompt, which keeps prompts short. The profile below is for *The Five Orange Pips* by Arthur Conan Doyle.

In [ ]:
BOOK = {
    "title": "The Five Orange Pips",
    "author": "Arthur Conan Doyle",

    "narrator": {"name": "Dr. John H. Watson", "gender": "male", "person": "first"},

    "style_notes": (
        "Precise, analytical, Victorian-confident. Watson narrates like a warm, grounded storyteller, "
        "never over-dramatic. Holmes speaks in short clipped bursts, dry and detached; his urgency is "
        "controlled, never shouted. Openshaw fragments under fear (short sentences, dashes). Tragedy lands "
        "quietly: understatement, short sentences, silence as punctuation. Break long Victorian sentences "
        "into natural Hinglish rhythm. Keep English place names (Baker Street, Waterloo Bridge, Horsham, "
        "Savannah). Holmes's guilt after Openshaw's death is understatement only, no melodrama. The ending "
        "(the Lone Star lost at sea) is flat and quiet, with no moral comment."
    ),

    "avoid": [
        "melodrama such as 'aankhon mein aansu' or 'dil dard se bhar gaya'",
        "heavy Sanskritised Hindi (adrooh, vishleshanatmak, prashasanik, uttam; say 'badhiya' or 'zabardast')",
        "yaar or bhai as forms of address",
        "tu / tujhe / tera / tune",
        "Gen-Z slang (sus, vibe, slay, no cap)",
        "dramatic Urdu narrator phrases like 'aur yahi tha unka anjaam'",
        "word-for-word Victorian sentence structure",
        "moral commentary that is not in the source",
    ],

    "characters": [
        {"name": "Sherlock Holmes", "role": "consulting detective",
         "voice": "precise, clipped, dry-witted, commanding; cold surface, warm core",
         "address": "clients say aap to him; he and Watson say tum to each other"},
        {"name": "Dr. John H. Watson", "role": "first-person narrator, Holmes's friend and chronicler",
         "voice": "warm, earnest, observant, understated; male verb forms (aaya, gaya, tha, dekha)",
         "address": "tum with Holmes, aap with clients"},
        {"name": "John Openshaw", "role": "young client, 22",
         "voice": "anxious, urgent, fragments under fear, quietly brave, honest",
         "address": "aap to Holmes and Watson"},
        {"name": "Colonel Elias Openshaw", "role": "dead uncle, ex-Confederate, KKK member",
         "voice": "fierce, paranoid, haunted; only described by others",
         "address": "third person only (woh, unka, unhone)"},
        {"name": "Joseph Openshaw", "role": "John's father, died in a chalk-pit",
         "voice": "dismissive, stubborn, outwardly brave, in denial",
         "address": "third person only (woh, unka, unhone)"},
    ],

    # Terms that stay in English exactly as written
    "keep": [
        "detective", "case", "clue", "gang", "suspect", "verdict", "evidence", "waterproof",
        "Scotland Yard", "telegram", "K.K.K.", "Ku Klux Klan", "Baker Street", "Waterloo Bridge",
        "Horsham", "Pondicherry", "Savannah", "Lone Star",
    ],

    # Fixed renderings: English term -> how it must appear in Hinglish
    "glossary": {
        "orange pips": "orange pips (santare ke sukhe beej, yahan dhamki ka nishan)",
        "five orange pips": "paanch sukhe orange pips (dhamki ka nishan)",
        "brass box": "peetal ka daabba",
        "Confederate": "Confederate (American Civil War mein South ki taraf se ladne wale)",
        "Reconstruction": "Reconstruction (Civil War ke baad South ko dobara banane ka daur, 1865-1877)",
        "carpet-bag politicians": "carpet-bag politicians (Civil War ke baad North se aaye neta)",
        "barque": "barque (do-teen mast wala sailing ship)",
        "equinoctial gales": "equinoctial gales (September-March ke mausami toofan)",
        "pince-nez": "pince-nez (naak par tikne wala chashma)",
        "sundial": "sundial (dhoop se time batane wali purani ghadi)",
        "lumber-room": "lumber-room (ghar ka attic)",
        "Lloyd's registers": "Lloyd's registers (jahazon ka official record)",
        "coroner's jury": "coroner's jury (sandehjanak maut ki jaanch karne wale log)",
        "stevedore": "stevedore (dock par maal chadhane-utaarne wala)",
        "deduction": "deduction (logical reasoning)",
        "draughts": "draughts (checkers, board game)",
        "backgammon": "backgammon (board game)",
        "implacable": "jo rukne wala nahi, ruthless",
        "inexorable": "aisi force jise roka nahi ja sakta",
        "conjecture": "andaza, educated guess",
    },
}
log.info("book profile: %s | %d characters | %d keep terms | %d glossary entries",
         BOOK["title"], len(BOOK["characters"]), len(BOOK["keep"]), len(BOOK["glossary"]))

## 4. Source text

Loads the file (uploading it first when running on Colab and it is missing) and strips what is not part of the story: leftover headers from earlier runs, page numbers, running page headers, hyphenated line breaks.

In [ ]:
def clean_source(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"^TRANSLATED TO ENGLISH\b.*?(?:={10,}|-{10,})\n+", "", text, flags=re.S)  # header from an earlier step
    text = re.sub(r"([A-Za-z])-\n([A-Za-z])", r"\1\2", text)      # word split across lines: "inter-\nesting"
    text = text.replace("\u00ad", "")                               # soft hyphens
    text = re.sub(r"(?m)^\s*Page\s+\d{1,4}\s*$", "", text)          # "Page 12"
    text = re.sub(r"(?m)^\s*\d{1,4}\s*$", "", text)                 # bare page numbers
    text = re.sub(r"(?m)^[A-Z][A-Za-z ',\-]{3,}\s+by\s+[A-Z][A-Za-z .]{2,}\s+\d{1,4}\s*$", "", text)  # "Title by Author 12"
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_source_path():
    if Path(SOURCE_FILE).exists():
        return SOURCE_FILE
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f"{SOURCE_FILE} not found - set SOURCE_FILE to the path of your .txt file")
    uploaded = files.upload()
    return next(iter(uploaded))


SOURCE_FILE = get_source_path()
raw_text = Path(SOURCE_FILE).read_text(encoding="utf-8-sig")
SOURCE_TEXT = clean_source(raw_text)
log.info("source: %s | %s words after cleaning (%s before)",
         SOURCE_FILE, f"{len(SOURCE_TEXT.split()):,}", f"{len(raw_text.split()):,}")
log.info("preview: %r", SOURCE_TEXT[:150])

## 5. Chunking

Rules, in order:

1. Split the text into blocks on blank lines and undo hard line wraps inside a paragraph. Short-line blocks (poems, lists) keep their line breaks.
2. Headings such as "CHAPTER I" always start a new chunk.
3. Pack whole paragraphs until the chunk would exceed `CHUNK_WORDS`. A single oversized paragraph is cut at sentence boundaries.
4. Each chunk also carries the last `OVERLAP_SENTENCES` sentences of the previous chunk as **context**. The model reads the context but must not translate it.

The sentence splitter does not break at abbreviations (Mr., Dr.), initials (S. H.), decimals or ellipses, and it keeps closing quotation marks attached to their sentence.

In [ ]:
ABBREVIATIONS = ("Mr", "Mrs", "Ms", "Dr", "Prof", "St", "Capt", "Col", "Gen", "Lt", "Sgt", "Rev", "Mt", "Jr", "Sr", "vs", "etc")
_ABBREV_RE = re.compile(r"\b(" + "|".join(ABBREVIATIONS) + r")\.")
_INITIAL_RE = re.compile(r"\b(?![IA]\.)([B-HJ-Z])\.(?=\s)")        # "S. H. for J. O."
_SENTENCE_END_RE = re.compile(r"""(?:(?<=[.!?])|(?<=[.!?]["'”’)\]]))\s+(?=["“‘'(]?[A-Z])""")
HEADING_RE = re.compile(
    r"^(?:(?:CHAPTER|Chapter)\s+[\dIVXLCDMivxlcdm]+\b"
    r"|(?:PART|Part|BOOK|Book|VOLUME|Volume)\s+\w+"
    r"|(?:ACT|SCENE)\s+[IVX\d]"
    r"|(?:PROLOGUE|EPILOGUE|PREFACE|INTRODUCTION|APPENDIX|CONCLUSION|FOREWORD|AFTERWORD)\b"
    r"|[IVX]{1,6}\.\s+\S)"
)


def split_sentences(text):
    """Split prose into sentences. Abbreviations, initials, decimals and ellipses do not end a sentence."""
    dot = "\x00"
    t = _ABBREV_RE.sub(lambda m: m.group(1) + dot, text)
    t = _INITIAL_RE.sub(lambda m: m.group(1) + dot, t)
    t = re.sub(r"(\d)\.(\d)", lambda m: m.group(1) + dot + m.group(2), t)
    t = re.sub(r"\.{2,}", lambda m: dot * len(m.group()), t)
    return [p.replace(dot, ".").strip() for p in _SENTENCE_END_RE.split(t) if p.strip()]


def split_long_sentence(sentence, max_words):
    """Last resort for a single sentence longer than a whole chunk: cut at a comma or semicolon."""
    words = sentence.split()
    if len(words) <= max_words:
        return [sentence]
    pieces, current = [], []
    for w in words:
        current.append(w)
        if (len(current) >= max_words * 0.7 and w.endswith((",", ";", ":"))) or len(current) >= max_words:
            pieces.append(" ".join(current))
            current = []
    if current:
        pieces.append(" ".join(current))
    return pieces


def to_blocks(text):
    """Split text into (kind, text) blocks where kind is 'heading', 'verse' or 'para'."""
    blocks = []
    for raw in re.split(r"\n\s*\n", text):
        lines = [ln.strip() for ln in raw.split("\n") if ln.strip()]
        if not lines:
            continue
        avg_words = sum(len(ln.split()) for ln in lines) / len(lines)
        if len(lines) > 2 and avg_words < 7:                 # poem or list: keep the line breaks
            blocks.append(("verse", "\n".join(lines)))
            continue
        para = " ".join(lines)                               # undo hard wraps
        is_heading = len(para.split()) <= 12 and HEADING_RE.match(para)
        blocks.append(("heading" if is_heading else "para", para))
    return blocks


def pack_sentences(sentences, max_words):
    """Group sentences into pieces of at most max_words (used for oversized paragraphs)."""
    pieces, current, count = [], [], 0
    for s in sentences:
        for part in split_long_sentence(s, max_words):
            w = len(part.split())
            if current and count + w > max_words:
                pieces.append(" ".join(current))
                current, count = [], 0
            current.append(part)
            count += w
    if current:
        pieces.append(" ".join(current))
    return pieces


def make_chunks(text, max_words=CHUNK_WORDS, overlap=OVERLAP_SENTENCES):
    """Return a list of {'id', 'text', 'context'} dicts, in book order."""
    pieces = []
    for kind, body in to_blocks(text):
        if kind == "para" and len(body.split()) > max_words:
            pieces += [("para", p) for p in pack_sentences(split_sentences(body), max_words)]
        else:
            pieces.append((kind, body))

    texts, current, count = [], [], 0
    for kind, body in pieces:
        w = len(body.split())
        heading_only = len(current) == 1 and current[0][0] == "heading"      # keep a heading with what follows it
        starts_new = kind == "heading" and current
        too_big = current and not heading_only and count + w > max_words
        if starts_new or too_big:
            texts.append("\n\n".join(b for _, b in current))
            current, count = [], 0
        current.append((kind, body))
        count += w
    if current:
        texts.append("\n\n".join(b for _, b in current))
    if len(texts) > 1 and len(texts[-1].split()) < max_words * 0.25:        # avoid a tiny last chunk
        texts[-2] += "\n\n" + texts.pop()

    chunks = []
    for i, body in enumerate(texts):
        context = ""
        if i and overlap:
            context = " ".join(split_sentences(texts[i - 1])[-overlap:])
        chunks.append({"id": i + 1, "text": body, "context": context})
    return chunks


CHUNKS = make_chunks(SOURCE_TEXT)
sizes = [len(c["text"].split()) for c in CHUNKS]
log.info("chunks: %d | words per chunk: min %d, avg %d, max %d",
         len(CHUNKS), min(sizes), sum(sizes) // len(sizes), max(sizes))

## 6. Prompts

Two passes with different jobs:

- **Pass 1** is a faithful draft at low temperature. It only has to get the meaning across: no style, no improvements.
- **Pass 2** rewrites that draft as natural Hinglish at a higher temperature. It sees the draft, a rolling summary of the story so far, the characters in the passage, the glossary terms that occur in it, a scene hint and a few examples of the target voice. It ends with a one-line `##PLOT_NOTE`, which feeds the "story so far" for later chunks.

Pass 2 is deliberately **not** shown the previous chunk's Hinglish output. Models that see their own recent output tend to repeat it, and continuity comes from the plot notes instead.

In [ ]:
VOICE_GUIDE = """
VOICE
Register: how a modern Indian novel or web series would narrate this. Mostly Hindi, English nouns where they are natural, a little Urdu where natural. No foul language. Emotion is felt but restrained, never melodramatic. Not stand-up comedy, not a WhatsApp forward, not a Bollywood script.
Code-switching: English verbs with Hindi grammar are fine ("manage karna", "handle karna", "realise hona"). Simple spoken words beat heavy formal Hindi. Keep everyday English words as they are (phone, office, train, police, station, ticket).
Address forms:
- elders, authority figures, strangers: aap / aapko / aapka (add "ji" or "sir" where natural)
- friends, equals, younger people: tum / tumhe / tumhara (elders may say beta / beti to children)
- never tu / tujhe / tera / tune unless the source clearly shows that rudeness
- never yaar or bhai as address, never abuse words, never Gen-Z slang
Gender: use the correct verb forms for each speaker (male: aaya, tha, gaya, bola; female: aayi, thi, gayi, boli).
""".strip()

# (English, Hinglish) pairs shown to Pass 2 as examples of the target voice.
FEW_SHOTS = [
    ("When I glance over my notes and records of the Sherlock Holmes cases between the years '82 and '90, "
     "I am faced by so many which present strange and interesting features that it is no easy matter to know "
     "which to choose and which to leave.",
     "Jab main '82 se '90 ke beech ke Sherlock Holmes ke notes palatta hoon, toh itne saare ajeeb aur "
     "interesting cases saamne aa jaate hain ki choose karna mushkil ho jaata hai. Kaunsa sunau, kaunsa chhodoon."),

    ("There is but one thing to do. It must be done at once. You must put this piece of paper which you have "
     "shown us into the brass box which you have described. You must also put in a note to say that all the "
     "other papers were burned by your uncle, and that this is the only one which remains.",
     "Sirf ek kaam karna hai. Abhi. Is kaagaz ko usi peetal ke daabbe mein daal do, jo tumne humein dikhaya tha. "
     "Saath mein ek note bhi likhna ki baaki saare papers tumhare uncle ne jala diye the. Yeh akela bacha tha."),

    ("\"Holmes,\" I cried, \"you are too late.\" \"Ah!\" said he, laying down his cup, \"I feared as much. How was "
     "it done?\" He spoke calmly, but I could see that he was deeply moved.",
     "\"Holmes,\" main chilla pada, \"bahut der ho gayi.\"\n"
     "\"Ah.\" Usne cup table par rakh diya. \"Mujhe bhi yahi darr tha. Yeh kaise hua?\" Usne shaant awaaz mein "
     "poochha. Lekin main jaanta tha, woh andar se toot gaya tha."),

    ("Opening it hurriedly, out there jumped five little dried orange pips, which pattered down upon his plate. "
     "I began to laugh at this, but the laugh was struck from my lips at the sight of his face. His lip had "
     "fallen, his eyes were protruding, his skin the colour of putty.",
     "Jaldi se khola, aur paanch chhote sukhe orange pips plate pe bikhar gaye. Pehle toh mujhe hassi aayi. "
     "Par phir unka chehra dekha, aur hassi wahin ruk gayi. Hont latke hue. Aankhein bahar nikli hui. Rang "
     "bilkul plaster jaisa."),

    ("He took an orange from the cupboard, and tearing it to pieces he squeezed out the pips upon the table. "
     "Of these he took five and thrust them into an envelope. On the inside of the flap he wrote \"S. H. for "
     "J. O.\" Then he sealed it and addressed it to \"Captain James Calhoun, Barque Lone Star, Savannah, Georgia.\"",
     "Holmes ne cupboard se ek orange nikala aur uske tukde kar diye. Beej table pe nichod diye. Paanch beej "
     "utha kar envelope mein daale. Envelope ke flap ke andar likha: \"S. H. for J. O.\" Phir seal kiya. "
     "Address: \"Captain James Calhoun, Barque Lone Star, Savannah, Georgia.\""),

    ("Sherlock Holmes sat for some time in silence, with his head sunk forward and his eyes bent upon the red "
     "glow of the fire. Then he lit his pipe, and leaning back in his chair he watched the blue smoke-rings as "
     "they chased each other up to the ceiling.",
     "Kuch der tak Sherlock Holmes chup raha. Sar thoda jhuka hua, aankhen angeethi ki laal anch pe. Phir pipe "
     "jalayi. Kursi mein dheere se tik gaya. Neeli smoke ki rings, ek ke baad ek, chhat ki taraf udti rahi. "
     "Woh sirf dekhta raha."),
]

# scene -> (keywords that suggest it, style hint for Pass 2). The first scene with the most hits wins.
SCENES = {
    "ACTION": (["charged", "attacked", "fired", "chased", "fled", "fight", "struck", "wounded", "escaped",
                "rushed", "dashed", "seized", "smashed", "pursuit", "pistol", "scrambled"],
               "Fast and punchy. Short sentences of 4-6 words. Raftaar mein likho, jaise scene chal raha ho."),
    "TENSION": (["crept", "tiptoed", "lurked", "peered", "locked", "warning", "danger", "trap", "ambush",
                 "suspicion", "uneasy", "dread", "footsteps", "froze", "whisper", "silence fell"],
                "Slow, cold, short sentences. Dread builds. Khamoshi bhaari, saans thami si."),
    "EMOTIONAL": (["wept", "cried", "sobbed", "tears", "grief", "sorrow", "mourning", "tragedy", "died",
                   "death", "farewell", "forgive", "embraced", "trembled", "gratitude", "remembered"],
                  "Slow and heavy. Let it breathe, let silences matter. Dheere bolo, dard ko jagah do."),
    "DEDUCTION": (["deduce", "observe", "infer", "conclude", "obvious", "clue", "evidence", "reasoning",
                   "logically", "therefore", "examined", "hypothesis", "elementary", "method", "footprint", "peculiar"],
                  "Precise and confident, clue by clue, each observation leading to the next, like a detective joining the dots."),
    "REVELATION": (["suddenly realised", "all became clear", "the truth", "secret", "revealed", "discovered",
                    "confession", "admitted", "at last", "unmasked", "exposed", "concealed", "turned out", "all along"],
                   "Short lines, a dramatic pause, the reader is shocked. Sach saamne aaya, ekdum."),
    "CONFRONTATION": (["how dare", "you lied", "deceived", "confess", "deny", "accused", "confronted", "guilty",
                       "cornered", "betrayed", "traitor", "fury", "rage", "demanded", "shouted", "ultimatum"],
                      "Direct and sharp, high stakes. Seedha aamna-saamna, koi peeche nahi hat raha."),
    "BANTER": (["laughed", "chuckled", "smiled", "grinned", "teased", "amusing", "absurd", "funny", "joke",
                "clever", "nonsense", "irony", "playful", "retorted", "quipped", "deadpan"],
               "Light and witty. Jaise dost ek doosre se hasi-mazaak kar rahe hon."),
    "HORROR": (["monster", "creature", "demon", "ghost", "horror", "terror", "nightmare", "corpse", "grave",
                "tomb", "darkness", "supernatural", "curse", "haunted", "scream", "coffin"],
               "Dark and creepy. Let the description scare, not the shouting. Thandi reedh."),
    "ROMANCE": (["desire", "passion", "longing", "beautiful", "blush", "admire", "proposal", "beloved",
                 "darling", "dearest", "kiss", "tender", "affection", "devotion", "courtship", "smitten"],
                "Warm and intimate. Feelings unfold slowly. Nazrein mili, kuch kaha nahi, par sab samajh aa gaya."),
    "PHILOSOPHICAL": (["meaning", "purpose", "existence", "fate", "destiny", "justice", "morality", "conscience",
                       "freedom", "wisdom", "contemplate", "ponder", "paradox", "illusion", "eternity", "mortality"],
                      "Deep but accessible. Simple words, heavy meaning. Gehri baat, seedhi zabaan mein."),
    "INNER_MONOLOGUE": (["i thought", "i wondered", "i knew", "i could not", "it seemed to me", "i asked myself",
                         "something inside", "was i wrong", "should i", "if only", "i feared", "i hoped", "i regretted"],
                        "Fragmented, honest, a little contradictory. Mann ke andar ki ulajh."),
    "DESCRIPTION": (["the room", "the house", "the garden", "the street", "sunset", "dawn", "storm", "wind",
                     "snow", "fog", "mountain", "river", "forest", "the sky", "the city", "moonlight"],
                    "Vivid and sensory. Take the reader there: rang, awaaz, khushboo."),
    "DAILY_LIFE": ([], "Conversational, easy flow. Aam zindagi ka scene, bilkul natural."),
}


def detect_scene(text):
    low = text.lower()
    scores = {name: sum(kw in low for kw in kws) for name, (kws, _) in SCENES.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else "DAILY_LIFE"

In [ ]:
def narrator_rule():
    n = BOOK.get("narrator")
    if not n:
        return ""
    if n.get("person") == "first":
        forms = "aaya / gaya / tha / dekha / bola" if n["gender"] == "male" else "aayi / gayi / thi / dekhi / boli"
        return (f"The narrator is {n['name']}, speaking in the first person ({n['gender']}). "
                f"Use {forms} verb forms for the narrator throughout.")
    return f"The narrator is {n['name']} (third person)."


def term_lines(text):
    """Keep-as-is terms and fixed renderings, limited to those that occur in this text."""
    low = text.lower()
    keep = [t for t in BOOK.get("keep", []) if t.lower() in low]
    fixed = [(k, v) for k, v in BOOK.get("glossary", {}).items() if k.lower() in low]
    lines = []
    if keep:
        lines.append("Keep exactly as written (do not translate): " + ", ".join(keep))
    if fixed:
        lines.append("Use these renderings: " + "; ".join(f'"{k}" -> "{v}"' for k, v in fixed))
    return "\n".join(lines)


def character_lines(text):
    """Characters mentioned in this text, with voice and address forms."""
    low = text.lower()
    lines = []
    for c in BOOK.get("characters", []):
        parts = [p for p in re.findall(r"[A-Za-z]+", c["name"]) if len(p) > 3]
        if any(p.lower() in low for p in parts):
            lines.append(f"- {c['name']} ({c['role']}). Voice: {c['voice']}. Address: {c['address']}.")
    return "\n".join(lines)


PASS1_SYSTEM = f"""You are a literary translator working from English into Hinglish (Hindi written in Roman script, with English words where Indians naturally use them).

This is a FAITHFUL FIRST DRAFT, not a polished version.
- Translate every sentence. Keep the meaning, the order and the structure.
- Do not add, remove, explain or improve anything.
- Roman script only. Never use Devanagari.
- Output only the translation: no notes, no labels, no commentary.
- Text inside <context> is the end of the previous passage, given only for continuity. Do not translate it.
{narrator_rule()}"""

_avoid = "; ".join(BOOK.get("avoid", []))
PASS2_SYSTEM = f"""You are a Hinglish literary editor. You receive a faithful draft translation of a passage from "{BOOK['title']}" by {BOOK['author']}. Rewrite it so that it reads as if it had originally been written in natural Hinglish (Hindi in Roman script, with English words where Indians naturally use them).

{VOICE_GUIDE}

BOOK STYLE
{BOOK['style_notes']}
{('Avoid: ' + _avoid) if _avoid else ''}
{narrator_rule()}

RULES
1. Keep the full meaning: every event, every line of dialogue and every detail in the draft must appear in your output.
2. Invent nothing: no new events, reactions, emotions, opinions, morals or commentary.
3. Rewrite at paragraph level, not sentence by sentence. Merge short sentences, split long awkward ones, and let the rhythm follow the scene.
4. Dialogue must sound like real speech and stay in character. Formal characters never turn casual.
5. Roman script only. Not one Devanagari character.
6. Use English only for proper nouns, everyday loanwords and the fixed terms you are given.
7. Output the rewritten text only: no labels, no notes, no preface.
8. After the last paragraph add one final line, exactly: ##PLOT_NOTE: <one English sentence saying what happens in this passage>"""

FEW_SHOT_TEXT = "\n\n".join(f"Example {i}\nEnglish: {en}\nHinglish: {hi}" for i, (en, hi) in enumerate(FEW_SHOTS, 1))


def build_pass1_user(chunk):
    parts = []
    if chunk["context"]:
        parts.append(f"<context>\n{chunk['context']}\n</context>")
    terms = term_lines(chunk["text"])
    if terms:
        parts.append(terms)
    parts.append("Translate this text into Hinglish:\n\n" + chunk["text"])
    return "\n\n".join(parts)


def build_pass2_user(chunk, draft, scene, story_notes):
    parts = []
    if story_notes:
        parts.append("STORY SO FAR (reference only, never output it):\n" + "\n".join(f"- {n}" for n in story_notes))
    chars = character_lines(chunk["text"])
    if chars:
        parts.append("CHARACTERS IN THIS PASSAGE:\n" + chars)
    terms = term_lines(chunk["text"])
    if terms:
        parts.append(terms)
    parts.append(f"SCENE FEEL (reference only): {SCENES[scene][1]}")
    parts.append("EXAMPLES OF THE TARGET VOICE:\n" + FEW_SHOT_TEXT)
    parts.append("DRAFT TO REWRITE:\n---\n" + draft + "\n---")
    parts.append("Rewrite the draft now. Keep every line of it. End with the ##PLOT_NOTE line.")
    return "\n\n".join(parts)


log.info("prompts ready: pass1 system %d chars, pass2 system %d chars, %d few-shot examples",
         len(PASS1_SYSTEM), len(PASS2_SYSTEM), len(FEW_SHOTS))

## 7. Model calls and validation

**Generation** streams tokens. Every 40 tokens the text so far is checked for a repetition loop (a block of three sentences that repeats), and generation is stopped early if one is found. The final stream event says whether the model stopped on its own or hit the token limit.

**Validation** returns a list of problems for each output. An empty list means the output is accepted.

| Problem | How it is detected |
|---------|--------------------|
| `empty` | fewer than 3 words |
| `hit_token_limit` | Ollama reports `done_reason == "length"` |
| `repetition_loop` | a block of three sentences appears twice (ignored if the source text itself repeats a block) |
| `devanagari` | any Devanagari character in the raw output (the prompt asks for Roman script only) |
| `length_ratio` | output words / source words outside 0.4 to 2.5 |
| `untranslated_english` | share of common English function words (the, and, was, with ...) above 10% |

The last check relies on a simple fact: Hinglish keeps English nouns but almost never English function words. Measured on the example pairs in this notebook, English source text scores 0.25 to 0.35 and the Hinglish versions score 0.00 to 0.03.

A failed attempt is retried with stricter sampling (higher `repeat_penalty`, lower temperature). When retries run out, the attempt with the fewest problems is kept.

In [ ]:
DEVANAGARI = re.compile(r"[\u0900-\u097F]")
ENGLISH_WORDS = frozenset(
    "the and of was were with that this have his her their they them from which would could "
    "there been she what when into over upon very but for not are".split()
)


def english_ratio(text):
    """Share of words that are common English function words. Near 0 for Hinglish, 0.25+ for English."""
    words = re.findall(r"[a-z']+", text.lower())
    return sum(w in ENGLISH_WORDS for w in words) / max(len(words), 1)


def find_loop(text, window=3):
    """If a block of `window` sentences repeats, return how many sentences to keep (up to and including
    the first copy of the block). Return None when there is no loop."""
    sents = split_sentences(text)
    if sents and not text.rstrip().endswith((".", "!", "?", '"', "”", "’", "'")):
        sents = sents[:-1]                                   # last sentence is still being written
    idx = [i for i, s in enumerate(sents) if len(s.split()) >= 4]
    keys = [tuple(w for w in re.findall(r"\w+", sents[i].lower()) if len(w) > 3) for i in idx]
    seen = {}
    for j in range(len(keys) - window + 1):
        block = tuple(keys[j:j + window])
        if block in seen:
            return idx[seen[block] + window - 1] + 1
        seen[block] = j
    return None


def offset_after(text, sentences, n):
    """Character offset in `text` just after its first n sentences (None if they cannot be located)."""
    cursor = 0
    for s in sentences[:n]:
        pos = text.find(s, cursor)
        if pos < 0:
            return None
        cursor = pos + len(s)
    return cursor


def cut_loop(text):
    """Drop the repeated part of a looping output and keep what came before it."""
    keep = find_loop(text)
    if keep is None:
        return text
    cut = offset_after(text, split_sentences(text), keep)
    return text[:cut].strip() if cut else text


def generate(system, user, options, watch_loops=True):
    """Stream one completion from Ollama. Returns (text, info). Stops early if the model starts looping."""
    opts = {**options, "num_ctx": NUM_CTX, "num_predict": NUM_PREDICT}
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    parts, events = [], 0
    info = {"truncated": False, "looped": False}
    stream = client.chat(model=MODEL, messages=messages, options=opts, stream=True)
    try:
        for event in stream:
            parts.append(event["message"]["content"])
            events += 1
            if event.get("done_reason") == "length":
                info["truncated"] = True
            if watch_loops and events % 40 == 0 and find_loop("".join(parts)) is not None:
                info["looped"] = True
                log.warning("  repetition loop detected after ~%d words - stopping early",
                            len("".join(parts).split()))
                break
    finally:
        stream.close()
    return "".join(parts), info


def validate(source, output, info, raw=None):
    """Return a list of problems found in a model output (empty list = good)."""
    n_out, n_src = len(output.split()), len(source.split())
    if n_out < 3:
        return ["empty"]
    problems = []
    if info["truncated"]:
        problems.append("hit_token_limit")
    source_repeats = find_loop(source) is not None            # a refrain in the source is not a model loop
    if not source_repeats and (info["looped"] or find_loop(output) is not None):
        problems.append("repetition_loop")
    if DEVANAGARI.search(raw if raw is not None else output):
        problems.append("devanagari")
    if n_src >= 10:
        ratio = n_out / n_src
        if not LENGTH_RATIO_RANGE[0] <= ratio <= LENGTH_RATIO_RANGE[1]:
            problems.append(f"length_ratio={ratio:.2f}")
    eng = english_ratio(output)
    if eng > MAX_ENGLISH_RATIO:
        problems.append(f"untranslated_english={eng:.2f}")
    return problems

In [ ]:
CHATTER_PATTERNS = [
    r"here(?:'s| is) the .*(?:translation|polished|hinglish|rewrite|rewritten|draft).*",
    r"hope (?:this|that) helps.*",
    r"(?:please find|let me know).*",
    r"as an ai\b.*",
    r"i cannot provide.*",
    r"errorhandler\b.*",                              # artifact seen in earlier Gemma runs
    r"[=\-*_\u2500-\u257F]{3,}",                      # divider lines echoed from the prompt
]
CHATTER_RE = re.compile(r"^\s*(?:" + "|".join(CHATTER_PATTERNS) + r")\s*$", re.I)
LABEL_RE = re.compile(r"^(?:Translation|Hinglish Translation|Hinglish|Output)\s*:\s*", re.I | re.M)
LONE_FRAGMENT_RE = re.compile(r"[a-z]{2,8}\.")        # a stray lowercase word like "imalik." on its own line
TYPO_FIXES = {                                        # recurring Gemma glitches seen in earlier runs
    r"\bwohne\b": "usne",
    r"\bletah\b": "leta",
    r"\bwoh ne\b": "usne",
}


def clean_output(text):
    """Remove model artifacts: think tags, code fences, labels, chatter lines, Devanagari, stray fragments."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    text = DEVANAGARI.sub("", text)
    text = text.replace("```", "")
    text = LABEL_RE.sub("", text)
    for pattern, replacement in TYPO_FIXES.items():
        text = re.sub(pattern, replacement, text)
    lines = [ln.rstrip() for ln in text.split("\n")
             if not CHATTER_RE.match(ln) and not LONE_FRAGMENT_RE.fullmatch(ln.strip())]
    text = "\n".join(lines)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def extract_plot_note(text):
    """Split the trailing '##PLOT_NOTE: ...' line off the model output. Returns (text, note)."""
    m = re.search(r"##\s*PLOT_?NOTE\s*:?\s*(.*)", text, re.I)
    if not m:
        return text, ""
    note = re.sub(r"^[\[\(*]+|[\]\)*]+$", "", m.group(1).split("\n")[0].strip())[:250]
    return text[:m.start()].strip(), note


def run_pass(label, system, user, options, source_text, retries):
    """Call the model until the output validates or retries run out. Returns the best attempt."""
    best = None
    watch = find_loop(source_text) is None
    for attempt in range(retries + 1):
        tweak = RETRY_TWEAKS[min(attempt, len(RETRY_TWEAKS) - 1)]
        started = time.time()
        raw, info = generate(system, user, {**options, **tweak}, watch_loops=watch)
        raw, note = extract_plot_note(raw)
        text = clean_output(raw)
        problems = validate(source_text, text, info, raw)
        if "repetition_loop" in problems:
            text = cut_loop(text)                              # keep what came before the repeats
        log.log(logging.WARNING if problems else logging.INFO,
                "  %s attempt %d: %.0fs, %d words%s", label, attempt + 1, time.time() - started,
                len(text.split()), f" | problems: {', '.join(problems)}" if problems else "")
        candidate = {"text": text, "note": note, "problems": problems, "attempt": attempt}
        if best is None or len(problems) <= len(best["problems"]):   # a later attempt wins ties
            best = candidate
        if not problems:
            break
    return best

## 8. Post-processing

Rule-based cleanup applied to every chunk after generation:

1. **Overlap trim**: if the model re-translated the context sentences, drop them from the top of the chunk (compared against the end of the previous chunk's translation; skipped for chunks that start a new chapter, and always logged).
2. **De-duplication**: drop a third identical line in a row, and drop a paragraph that nearly repeats one of the last few paragraphs.
3. **Narrator gender**: for a first-person narrator, fix verb forms that disagree with the profile (`main gayi` becomes `main gaya`). Quoted speech is left alone.
4. **Proper nouns**: restore the capitalisation of character names and capitalised glossary terms.
5. **Spacing**: tidy blank lines and spaces before punctuation.

After that, any paragraph that is still half-translated (too many English function words) goes through one small LLM repair call. Nothing is ever deleted for being English: if the repair does not help, the paragraph stays and the chunk is reported.

In [ ]:
def norm(text):
    return re.sub(r"\W+", " ", text.lower()).strip()


def trim_overlap(text, prev_text, threshold=0.70):
    """Drop sentences at the top of `text` that repeat the end of the previous chunk's translation."""
    if not prev_text:
        return text
    prev_tail = [norm(s) for s in split_sentences(prev_text)[-3:]]
    sents = split_sentences(text)
    cut = 0
    for i, s in enumerate(sents[:3]):
        if len(s.split()) >= 5 and any(SequenceMatcher(None, norm(s), p).ratio() >= threshold for p in prev_tail):
            cut = i + 1
    if not cut:
        return text
    offset = offset_after(text, sents, cut)
    rest = text[offset:].strip() if offset else ""
    return rest if len(rest.split()) >= 0.3 * len(text.split()) else text    # never throw away most of a chunk


def dedupe_lines(text, max_repeat=2):
    """Allow a line to appear at most `max_repeat` times in a row."""
    out, last, repeats = [], None, 0
    for line in text.split("\n"):
        key = norm(line)
        if key and key == last:
            repeats += 1
            if repeats >= max_repeat:
                continue
        elif key:
            last, repeats = key, 0
        out.append(line)
    return "\n".join(out)


def dedupe_paragraphs(text, window=8, threshold=0.85):
    """Drop a paragraph that nearly repeats one of the previous few paragraphs. Short ones are never dropped."""
    kept, keys = [], []
    for para in text.split("\n\n"):
        key = norm(para)
        if len(key.split()) >= 8 and any(SequenceMatcher(None, key, k).ratio() >= threshold for k in keys[-window:]):
            continue
        kept.append(para)
        keys.append(key)
    return "\n\n".join(kept)


_TO_MALE = {"aayi": "aaya", "gayi": "gaya", "thi": "tha", "boli": "bola", "chali": "chala", "rahi": "raha", "hui": "hua"}
_TO_FEMALE = {v: k for k, v in _TO_MALE.items()}


def fix_narrator_gender(text):
    """Fix 'main <verb>' agreement for a first-person narrator. Text inside quotation marks is not touched."""
    n = BOOK.get("narrator") or {}
    if n.get("person") != "first":
        return text
    mapping = _TO_MALE if n.get("gender") == "male" else _TO_FEMALE
    main_re = re.compile(r"\b(main)\s+(" + "|".join(mapping) + r")\b", re.I)
    laga = ("lagi", "laga") if n.get("gender") == "male" else ("laga", "lagi")
    laga_re = re.compile(rf"\b(mujhe)\s+{laga[0]}\b", re.I)
    pieces = re.split(r'("[^"\n]*"|“[^”\n]*”)', text)          # odd indices are quoted speech
    for i in range(0, len(pieces), 2):
        pieces[i] = main_re.sub(lambda m: f"{m.group(1)} {mapping[m.group(2).lower()]}", pieces[i])
        pieces[i] = laga_re.sub(lambda m: f"{m.group(1)} {laga[1]}", pieces[i])
    return "".join(pieces)


_TITLES = {"Dr", "Mr", "Mrs", "Ms", "Miss", "Colonel", "Captain", "Sir", "Lord", "Lady", "Inspector"}


def restore_names(text):
    """Restore the capitalisation of character names and capitalised glossary terms."""
    names = set()
    for c in BOOK.get("characters", []):
        names.add(c["name"])
        names.update(w for w in re.findall(r"[A-Za-z']+", c["name"]) if len(w) > 3 and w not in _TITLES)
    names.update(t for t in BOOK.get("keep", []) + list(BOOK.get("glossary", {})) if t[:1].isupper())
    if not names:
        return text
    canonical = {n.lower(): n for n in names}
    pattern = re.compile(r"(?<!\w)(" + "|".join(re.escape(n) for n in sorted(names, key=len, reverse=True)) + r")(?!\w)", re.I)
    return pattern.sub(lambda m: canonical[m.group(1).lower()], text)


def tidy_spacing(text):
    text = "\n".join(ln.rstrip() for ln in text.split("\n"))
    text = re.sub(r"[ \t]+([,.!?;:])", r"\1", text)
    text = re.sub(r"[ \t]{2,}", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def postprocess(text, prev_translation=""):
    cleaned = trim_overlap(text, prev_translation)
    if len(cleaned.split()) < len(text.split()):
        log.info("  trimmed %d words of repeated context from the top of the chunk",
                 len(text.split()) - len(cleaned.split()))
    cleaned = dedupe_lines(cleaned)
    cleaned = dedupe_paragraphs(cleaned)
    cleaned = fix_narrator_gender(cleaned)
    cleaned = restore_names(cleaned)
    cleaned = tidy_spacing(cleaned)
    return cleaned or text


REPAIR_SYSTEM = """You fix half-translated Hinglish. The paragraph you receive is Hinglish in Roman script in which some English sentences or clauses were left untranslated.
Translate only those English parts into natural Hinglish. Leave everything that is already Hinglish unchanged.
Keep the same tone and register. Keep proper nouns and everyday English loanwords as they are.
Roman script only. Output the corrected paragraph and nothing else."""


def repair_english(text):
    """Re-translate paragraphs that still contain a lot of English. Returns (text, number_repaired)."""
    paras = text.split("\n\n")
    repaired = 0
    for i, para in enumerate(paras):
        before = english_ratio(para)
        if len(para.split()) < 8 or before <= REPAIR_ENGLISH_RATIO:
            continue
        fixed, _ = generate(REPAIR_SYSTEM, para, REPAIR_OPTIONS, watch_loops=False)
        fixed = clean_output(fixed)
        length_ok = 0.6 <= len(fixed.split()) / len(para.split()) <= 1.5
        if fixed and length_ok and english_ratio(fixed) < before:
            paras[i] = fixed
            repaired += 1
        else:
            log.warning("  paragraph still has English (ratio %.2f); repair did not help, kept as is", before)
    return "\n\n".join(paras), repaired

## 9. Translating one chunk

```
Pass 1 (retry on problems)  ->  Pass 2 (retry on problems)  ->  post-process  ->  repair half-translated paragraphs
```

Every chunk ends up with one of four statuses, stored in the checkpoint:

| Status | Meaning |
|--------|---------|
| `ok` | Pass 2 output passed validation |
| `flagged` | retries ran out; the best attempt is kept and its problems are listed in the report |
| `draft_only` | Pass 2 failed badly (empty or still English), so the Pass 1 draft is used instead |
| `failed` | an exception such as a lost server connection; the chunk is retried on the next run |

In [ ]:
def first_sentence(text, limit=120):
    for s in split_sentences(text):
        if len(s.split()) > 6:
            return s[:limit]
    return " ".join(text.split()[:25])


def translate_chunk(chunk, story_notes, prev_translation):
    """Run both passes on one chunk and return a checkpoint record."""
    scene = detect_scene(chunk["text"])
    log.info("chunk %d/%d | %d words | scene=%s", chunk["id"], len(CHUNKS), len(chunk["text"].split()), scene)

    p1 = run_pass("pass1", PASS1_SYSTEM, build_pass1_user(chunk), PASS1_OPTIONS, chunk["text"], MAX_RETRIES)
    if "empty" in p1["problems"]:
        raise RuntimeError("pass 1 returned no text")
    draft = p1["text"]

    p2 = run_pass("pass2", PASS2_SYSTEM, build_pass2_user(chunk, draft, scene, story_notes),
                  PASS2_OPTIONS, chunk["text"], MAX_RETRIES)
    text, problems = p2["text"], p2["problems"]

    has = lambda found, name: any(p.startswith(name) for p in found)    # problems may carry a value: "name=0.31"
    pass2_unusable = has(problems, "empty") or (
        has(problems, "untranslated_english") and not has(p1["problems"], "untranslated_english"))
    if pass2_unusable:
        log.warning("  pass 2 unusable (%s) - using the pass 1 draft", ", ".join(problems))
        text, status = draft, "draft_only"
    else:
        status = "flagged" if problems else "ok"

    starts_section = bool(HEADING_RE.match(chunk["text"]))
    text = postprocess(text, "" if starts_section else prev_translation)
    text, repaired = repair_english(text)
    if repaired:
        log.info("  repaired %d half-translated paragraph(s)", repaired)

    return {"id": chunk["id"], "status": status, "translation": text, "problems": problems, "scene": scene,
            "plot_note": p2["note"] or first_sentence(chunk["text"]), "attempts": p2["attempt"] + 1}

## 10. Checkpoint and resume

Each finished chunk is appended as one JSON line to `<name>.checkpoint.jsonl`, and the file is flushed immediately. A small `<name>.meta.json` stores a fingerprint of the cleaned source text and the chunking settings.

When the run starts:

- If a checkpoint exists, chunks with status `ok`, `flagged` or `draft_only` are skipped, and only missing or `failed` chunks are translated.
- The "story so far" notes and the previous chunk's translation are rebuilt from the checkpoint, so a resumed run continues exactly where the last one stopped.
- If the source text or the chunking settings have changed, the run refuses to resume (the chunk numbers would no longer match). Restore the old settings, or set `RESTART = True`.

If a Colab runtime is reset, upload the `.checkpoint.jsonl` and `.meta.json` files back into `OUTPUT_DIR` before running. Saving `OUTPUT_DIR` on Google Drive avoids this.

In [ ]:
STEM = Path(SOURCE_FILE).stem
CHECKPOINT_PATH = OUTPUT_DIR / f"{STEM}.checkpoint.jsonl"
META_PATH = OUTPUT_DIR / f"{STEM}.meta.json"
OUTPUT_PATH = OUTPUT_DIR / f"{STEM}_hinglish.txt"
REPORT_PATH = OUTPUT_DIR / f"{STEM}_report.txt"
DONE_STATUSES = {"ok", "flagged", "draft_only"}


def fingerprint():
    return {"source_sha1": hashlib.sha1(SOURCE_TEXT.encode("utf-8")).hexdigest(),
            "chunk_words": CHUNK_WORDS, "overlap_sentences": OVERLAP_SENTENCES, "model": MODEL}


def load_checkpoint():
    """Return {chunk_id: record} from an earlier run, or {} for a fresh start."""
    fp = fingerprint()
    if RESTART:
        for path in (CHECKPOINT_PATH, META_PATH):
            path.unlink(missing_ok=True)
        log.info("RESTART=True: old checkpoint removed")
    if META_PATH.exists():
        old = json.loads(META_PATH.read_text(encoding="utf-8"))
        changed = [k for k in ("source_sha1", "chunk_words", "overlap_sentences") if old.get(k) != fp[k]]
        if changed:
            raise RuntimeError(f"Checkpoint was made with different settings ({', '.join(changed)}). "
                               "Restore them, or set RESTART = True to start over.")
    else:
        META_PATH.write_text(json.dumps(fp, indent=2), encoding="utf-8")

    records = {}
    if CHECKPOINT_PATH.exists():
        for line in CHECKPOINT_PATH.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:                  # a line cut off when the session died
                log.warning("ignoring a corrupt line in the checkpoint")
                continue
            records[rec["id"]] = rec                      # later lines win
    return records


def save_record(rec):
    with open(CHECKPOINT_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        f.flush()

## 11. Run

Safe to stop at any time (interrupt the cell) and run again: finished chunks are not repeated. The loop also stops by itself when `TIME_LIMIT_MIN` is about to be exceeded, or after three failures in a row (usually a dead Ollama server).

In [ ]:
def fmt_duration(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f"{h}h {m:02d}m" if h else f"{m}m {s:02d}s"


def run():
    records = load_checkpoint()
    notes = deque((records[i]["plot_note"] for i in sorted(records) if records[i].get("plot_note")),
                  maxlen=STORY_NOTES)
    todo = [c for c in CHUNKS if records.get(c["id"], {}).get("status") not in DONE_STATUSES]
    log.info("chunks: %d total | %d already done | %d to do", len(CHUNKS), len(CHUNKS) - len(todo), len(todo))

    started, durations, failures = time.time(), [], 0
    try:
        for n, chunk in enumerate(todo):
            if TIME_LIMIT_MIN and durations:
                projected = time.time() - started + sum(durations) / len(durations)
                if projected > TIME_LIMIT_MIN * 60:
                    log.warning("time limit of %d min reached - stopping. Run this cell again to resume.", TIME_LIMIT_MIN)
                    break
            chunk_started = time.time()
            prev = records.get(chunk["id"] - 1, {}).get("translation", "")
            try:
                rec = translate_chunk(chunk, list(notes), prev)
                failures = 0
            except Exception as exc:
                failures += 1
                log.error("chunk %d failed: %s", chunk["id"], exc)
                rec = {"id": chunk["id"], "status": "failed", "translation": "", "problems": [str(exc)],
                       "plot_note": "", "scene": "", "attempts": 0}
            rec["seconds"] = round(time.time() - chunk_started, 1)
            save_record(rec)
            records[chunk["id"]] = rec
            if rec["status"] != "failed" and rec["plot_note"]:
                notes.append(rec["plot_note"])

            durations.append(rec["seconds"])
            eta = sum(durations) / len(durations) * (len(todo) - n - 1)
            log.info("chunk %d saved [%s] | %d words | %s | elapsed %s | ETA %s",
                     chunk["id"], rec["status"], len(rec["translation"].split()), fmt_duration(rec["seconds"]),
                     fmt_duration(time.time() - started), fmt_duration(eta))
            if SHOW_PREVIEW and rec["translation"]:
                log.info("  preview: %s", rec["translation"][:120].replace("\n", " "))
            if failures >= 3:
                log.error("3 chunks failed in a row - is the Ollama server still running? Stopping.")
                break
    except KeyboardInterrupt:
        log.warning("interrupted - finished chunks are saved. Run this cell again to resume.")

    counts = Counter(r["status"] for r in records.values())
    log.info("status: %s | not yet translated: %d",
             ", ".join(f"{k}={v}" for k, v in sorted(counts.items())) or "none",
             sum(1 for c in CHUNKS if c["id"] not in records))
    return records


RECORDS = run()

## 12. Output

Stitches the chunk translations in book order and runs a final cleanup across chunk boundaries (repeated paragraphs, name capitalisation, spacing). A chunk that has no translation is replaced by a visible `[[ chunk N was not translated ]]` marker rather than silently leaving a gap. The QA report lists every chunk that is not plainly `ok`.

In [ ]:
def assemble_book(records):
    parts, missing = [], []
    for c in CHUNKS:
        rec = records.get(c["id"])
        if rec and rec["translation"].strip():
            parts.append(rec["translation"])
        else:
            missing.append(c["id"])
            parts.append(f"[[ chunk {c['id']} was not translated ]]")
    book = tidy_spacing(restore_names(dedupe_paragraphs("\n\n".join(parts))))
    return book, missing


def write_report(records, path):
    lines = []
    for c in CHUNKS:
        rec = records.get(c["id"])
        if rec is None:
            lines.append(f"chunk {c['id']}: NOT TRANSLATED (never ran)")
        elif rec["status"] != "ok":
            lines.append(f"chunk {c['id']}: {rec['status'].upper()} | {', '.join(rec['problems']) or '-'}")
    path.write_text("\n".join(lines) + "\n" if lines else "All chunks passed validation.\n", encoding="utf-8")
    return lines


book, missing = assemble_book(RECORDS)
OUTPUT_PATH.write_text(book, encoding="utf-8")
issues = write_report(RECORDS, REPORT_PATH)

log.info("output: %s | %s words from %s source words",
         OUTPUT_PATH, f"{len(book.split()):,}", f"{len(SOURCE_TEXT.split()):,}")
log.info("report: %s | %d chunk(s) need a look", REPORT_PATH, len(issues))
for line in issues[:10]:
    log.warning("  %s", line)
if missing:
    log.warning("%d chunk(s) missing from the output: %s - run section 11 again to finish them", len(missing), missing)

### Download

Downloads the translation and the checkpoint files (keep the checkpoint if you may want to resume or re-run later).

In [ ]:
try:
    from google.colab import files
    for path in (OUTPUT_PATH, REPORT_PATH, CHECKPOINT_PATH, META_PATH):
        files.download(str(path))
except ImportError:
    log.info("not running on Colab - the files are in %s", OUTPUT_DIR.resolve())

## Design notes

**Why two passes.** One pass has to translate and write well at the same time, and at low temperature it reads like a translation while at high temperature it drops details. Splitting the job lets Pass 1 stay faithful and Pass 2 concentrate on voice, with the draft as an anchor against invention.

**Why chunks with context.** Gemma 3 27B on a T4 is slow and loses detail on long inputs. About 350 words per chunk keeps each call to a minute or two. The previous chunk's last sentences are given as read-only context so a chunk does not start cold, and Pass 2 gets short plot notes for the wider story.

**Why validate and retry.** Local models fail in a few recognisable ways: repeating a block of text, answering in Devanagari, leaving English untranslated, stopping early, or running on. Each has a cheap check, and retrying with a higher `repeat_penalty` fixes most of them. When retries run out the best attempt is kept and the chunk is flagged rather than dropped.

**Why a checkpoint file.** A full book takes hours and Colab sessions end. An append-only file of finished chunks is the simplest thing that makes the run restartable, and it is easy to inspect or edit by hand.

**Known limits.**
- The checks are rules, not a quality score. They catch broken output, not mediocre Hinglish, so a human read of a sample is still needed.
- Scene detection is keyword counting and is only a style hint.
- Sentence splitting is regex-based and can be wrong on unusual punctuation.
- The English-ratio check is tuned for English prose and will be less reliable if a book quotes long passages in English on purpose.
- Gender and address forms (aap, tum, tu) are enforced by the prompt, not checked afterwards, apart from the narrator fix.